In [1]:
import ast
import inspect
import json
import os
import sys

sys.path.insert(0, ".")  # ensure repo root is on path
os.environ["DATA_HOME_DIR"] = os.path.abspath("metadata")
os.environ["STRUCTNL_MODE"] = "fretish"

import pandas as pd
import data_loader
from nl2structnl_fretish import get_structNL_prompt_simple

# Match the configuration in run_llm.ipynb exactly, so the prompts below
# are the EXACT prompts that run_llm.ipynb would send.
data_home_dir = "./benchmarks/fret_specs/"
dataset_name = "FSM-S"
row_idx = 0
model = "qwen2.5:32b"
num_trial = 1  # -> k passed to get_structNL_prompt_simple (see cell below)

df = pd.read_excel(data_home_dir + dataset_name + "/PlausibleSpecs.xlsx", engine="openpyxl")
input_nl = df.iloc[row_idx]["NL"]

['timing options', 'timing bool exps', 'scope 1 options', 'scope 1 bool exps', 'scope 2 options', 'scope 2 bool exps']


In [2]:
# Extract system_prompt / user_prompt construction directly from the
# live source of data_loader.generate_ap_dict_via_ollama, so this stays in
# sync even if that function's prompt text changes.
source = inspect.getsource(data_loader.generate_ap_dict_via_ollama)
tree = ast.parse(source)
func_node = tree.body[0]

ap_system_prompt = None
ap_user_prompt_prefix = None
for node in ast.walk(func_node):
    if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name):
        name = node.targets[0].id
        if name == "system_prompt":
            ap_system_prompt = ast.literal_eval(node.value)
        elif name == "user_prompt":
            value = node.value
            if isinstance(value, ast.BinOp):
                ap_user_prompt_prefix = ast.literal_eval(value.left)

# Real requirements list for dataset_name, exactly as generate_ap_dict_via_ollama
# computes it: df["NL"].dropna().tolist()
nl_requirements = df["NL"].dropna().tolist()

ap_user_prompt = ap_user_prompt_prefix + json.dumps(nl_requirements, indent=2)

print("=== SYSTEM PROMPT ===\n")
print(ap_system_prompt)
print("\n\n=== USER PROMPT ===\n")
print(ap_user_prompt)

=== SYSTEM PROMPT ===

You are an expert in requirements engineering and formal specification. Given a list of natural language requirements, identify all atomic boolean propositions (system state variables) needed to express them formally as FRETish requirements. For each proposition provide a variable name and a brief description. Variable names must be valid identifiers in the FRET requirements grammar: they must start with a letter and contain only letters, digits, and underscores (no spaces, hyphens, or other special characters), and must not be one of FRET's reserved words (e.g. shall, when, if, mode, and, or, not, true, false, until, within). Use lowercase snake_case names for ordinary variables (e.g. "sensor_is_active"). For a variable that represents a finite-state-machine being in a particular mode, use the pattern "state_is_<MODE_NAME>" with the mode name in upper case (e.g. "state_is_NOMINAL", "state_is_FAULT"). IMPORTANT: requirements often give the exact variable name to 

In [3]:
# ap_dict for dataset_name/row_idx, loaded the same way data_loader.load_vars()
# does when Variables.xlsx is present (always_generate=False).
# Note: run_llm.ipynb sets ALWAYS_GENERATE_VARIABLES=True, in which case ap_dict
# instead comes from the Ollama call shown in the cell above; this Variables.xlsx
# version is used here only so this cell can run without an Ollama call.
var_df = pd.read_excel(data_home_dir + dataset_name + "/Variables.xlsx", engine="openpyxl")
ap_dict = data_loader.get_ap_dict(var_df)

# get_nl2structnl_translation (called from run_llm.ipynb) is invoked with
# mode=None, dcmp=None, and k=num_trial -- forced to k=1 when mode is None.
formalization_system_prompt, formalization_user_prompt = get_structNL_prompt_simple(
    input_nl,
    ap_dict,
    dcmp=None,
    k=num_trial,
)

print("=== SYSTEM PROMPT ===\n")
print(formalization_system_prompt)
print("\n\n=== USER PROMPT ===\n")
print(formalization_user_prompt)

=== SYSTEM PROMPT ===

You are an expert in Linear Temporal Logic and requirements engineering. Your job is to translate natural language requirements to structured natural language that capture the intents of the requirements.


To produce the structured natural language property, you compose it from a set of templates.
If the chosen option contains boolean expression placeholders (i.e., bool_exp1, bool_exp2, bool_exp3, bool_exp4), you need to produce boolean expressions that will replace the placeholders.
Boolean expressions can only contain boolean operators (e.g., !, &, |, ->, <->) and can only atomic propositions (NO NUMERICAL COMPARISON OPERATORS ALLOWED)

The following lists the ONLY valid options for decision1, decision2, and decision3. You MUST copy one of these strings exactly — do not paraphrase or invent new values:

decision1_options = [
    "while bool_exp1, _ABSTRACT_VAR1_",
    "before bool_exp1, _ABSTRACT_VAR1_",
    "after bool_exp1, _ABSTRACT_VAR1_",
    "whenever bo